In [1]:
# Competition-Solution/notebooks/trial/00_data_profiling/02_data_profiling_class_value_counts.ipynb

---

##### <b>Imports</b>

In [2]:

import sys
from rich.console import Console

sys.path.append("../../../src")
import data_utils

console = Console()

##### <b>Loading the data</b>

In [3]:
df_call2action, df_fdgo, df_violence = data_utils.load_competition_data(data_dir="../../data/raw/", type="trial")

Loading competition data...

Data of type 'trial' loaded successfully.

##### <b>Class Value Counts</b>

In [4]:
print(f"Subtask 1 (Call2Action):")
print(df_call2action["C2A"].value_counts())
print(f"Subtask 2 (DBO):")
print(df_fdgo["DBO"].value_counts())
print(f"Subtask 3 (Violence):")
print(df_violence["VIO"].value_counts())

# Imbalance between the classes (value_counts_per_class / total_samples * 100)
c2a_imbalance = (df_call2action["C2A"].value_counts() / len(df_call2action)) * 100
print(c2a_imbalance)
dbo_imbalance = (df_fdgo["DBO"].value_counts() / len(df_fdgo)) * 100
print(dbo_imbalance)
vio_imbalance = (df_violence["VIO"].value_counts() / len(df_violence)) * 100
print(vio_imbalance)

Subtask 1 (Call2Action):
C2A
False    949
True     102
Name: count, dtype: int64
Subtask 2 (DBO):
DBO
nothing       921
criticism     122
agitation       6
subversive      4
Name: count, dtype: int64
Subtask 3 (Violence):
VIO
False    991
True      60
Name: count, dtype: int64
C2A
False    90.294957
True      9.705043
Name: count, dtype: float64
DBO
nothing       87.464387
criticism     11.585945
agitation      0.569801
subversive     0.379867
Name: count, dtype: float64
VIO
False    94.291151
True      5.708849
Name: count, dtype: float64


<u><p>Interpretation of the class value counts</p></u>

*   **Subtask 1 (Call2Action - Label: `C2A`):**
    *   `False`: 949 instances *(~90.3%)*
    *   `True`: 102 instances *(~9.7%)*
    *   *Observation:* **Significant class imbalance.**

*   **Subtask 2 (DBO - Attacks on Democratic Basic Order - Label: `DBO`):**
    *   `nothing`: 921 instances *(~87.5%)*
    *   `criticism`: 122 instances *(~11.6%)*
    *   `agitation`: 6 instances *(~0.6%)*
    *   `subversive`: 4 instances *(~0.4%)*
    *   *Observation:* **Extreme class imbalance.** <br>The `'nothing'` class heavily dominates. The target classes representing attacks (`'agitation'`, `'subversive'`) are very rare in this trial dataset. Even `'criticism'` is significantly underrepresented compared to 'nothing'. This is likely going to pose a significant challenge when tackling this particular subtask.

*   **Subtask 3 (Violence Detection - Label: `VIO`):**
    *   `False`: 991 instances *(~94.3%)*
    *   `True`: 60 instances *(~5.7%)*
    *   *Observation:* **High class imbalance.**

<u><p>Summary & Implication</p></u>

*   *Summary of Observations:* All three subtasks in the trial dataset show significant to extreme class imbalance. Especially the more critical "harmful" categories in DBO and VIO, are substantially underrepresented.
*   *Implication:*
    *   **Macro-F1-Score Codabench Competition Evaluation Metric:** Standard accuracy would have been a highly misleading performance indicator. **Macro-F1 Score** gives equal weight to each class. This better reflects the performance on the underrepresented minority classes identified above.

    *   A high F1-score requires both good precision and good recall. In tasks like harmful content detection, simply maximizing recall (finding all harmful instances) at the cost of very low precision (flagging many harmless instances as harmful) would lead to an unusable system, despite potentially yielding a decent F1 if precision isn't abysmal. The Macro-F1 encourages a balance across all classes.

    **Macro-F1-Score Formula:**

    The Macro-F1 score is calculated by first determining the F1-score for each individual class and then taking the unweighted average of these scores. This gives equal importance to each class, regardless of its frequency.

    1.  **For each class `i` (from 1 to C, where C is the number of classes):**

        *   **Precision_i:** Measures the accuracy of positive predictions for class `i`.
            $$
            \text{Precision}_i = \frac{TP_i}{TP_i + FP_i}
            $$
            Where:
            *   $TP_i$ (True Positives for class `i`): Number of instances correctly predicted as class `i`.
            *   $FP_i$ (False Positives for class `i`): Number of instances incorrectly predicted as class `i` (but were actually a different class).

        *   **Recall_i:** Measures how many of the actual positive instances for class `i` were correctly identified.
            $$
            \text{Recall}_i = \frac{TP_i}{TP_i + FN_i}
            $$
            Where:
            *   $FN_i$ (False Negatives for class `i`): Number of instances of class `i` that were incorrectly predicted as a different class.

        *   **Precision and Recall Visualized** (Sourced from *'Introduction to Information Retrieval'* Slides and slightly modified to accommodate for classification instead of IR task; *Lecture 5, Slide 16*)<br>
            <img src="../../assets/images/venn_diagram_precision_recall_modified.png" alt="Precision and Recall Venn-Diagram" width="500" height="300">


        *   **F1-Score_i:** The harmonic mean of Precision_i and Recall_i for class `i`. It provides a balance between the two.
            $$
            \text{F1-Score}_i = 2 \times \frac{\text{Precision}_i \times \text{Recall}_i}{\text{Precision}_i + \text{Recall}_i}
            $$
            If $\text{Precision}_i + \text{Recall}_i = 0$, then $\text{F1-Score}_i$ is typically defined as 0 to avoid division by zero.

    2.  **Macro-F1-Score:** The unweighted average of the F1-scores for all C classes.
        $$
        \text{Macro-F1} = \frac{1}{C} \sum_{i=1}^{C} \text{F1-Score}_i
        $$
        or written out as expanded sum:
        $$
        \text{Macro-F1} = \frac{\text{F1-Score}_1 + \text{F1-Score}_2 + \dots + \text{F1-Score}_C}{C}
        $$


    *   **Modeling Strategies:** The class imbalance will need to be handled in some way. The following strategies or a combination of them are possibilities:
        *   **Stratified data splitting**
        *   **Class weighting** in the loss function.
        *   **Resampling techniques** (e.g. oversampling minority classes, undersampling majority classes)
        *   **Data Augmentation** focused on minority class examples.
    *   *Note:* Due to the fact, that this is merely the trial data, more complex/time consuming approaches to handle this class imbalance will only be undertaken on the training data if need be.